# Classificador: handgrip ou não?

Classificação binária sobre os frames coletados em `/training`.
Classe positiva (`1`) = **handgrip**.

Roteiro baseado nos capítulos iniciais de *Mãos à Obra: Aprendizado de Máquina com Scikit-Learn, Keras & TensorFlow*:

| Seção | Capítulo |
|---|---|
| 1–3. Carregar, explorar e separar o teste | 2 – Projeto de ponta a ponta |
| 4. Features a partir dos pixels | 2 e 3 (MNIST) |
| 5–7. Baseline, `SGDClassifier` e métricas | 3 – Classificação |
| 8. Outros modelos | 4 (logística), 5 (SVM), 6/7 (árvores e florestas) |
| 9. PCA | 8 – Redução de dimensionalidade |
| 10–11. Ajuste fino e teste final | 2 |
| 12. Exportar | — |

Dados: `dataset/manifest.csv`, com as colunas `full_path, crop_path, label, session, has_hand, captured_at`. Os caminhos são relativos à raiz do repositório.

## 0. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from PIL import Image

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# Works whether Jupyter was started from the repo root or from ml/.
ROOT = Path.cwd().parent if Path.cwd().name == "ml" else Path.cwd()
DATASET_DIR = ROOT / "dataset"
MANIFEST_PATH = DATASET_DIR / "manifest.csv"
MODELS_DIR = ROOT / "ml" / "models"
MODELS_DIR.mkdir(exist_ok=True)

POSITIVE_LABEL = "handgrip"   # y = 1
IMAGE_SIZE = (32, 32)         # (width, height) after resizing; 32x32 grayscale -> 1024 features

print("scikit-learn", sklearn.__version__)
print("manifest:", MANIFEST_PATH, "found" if MANIFEST_PATH.exists() else "NOT FOUND — collect samples at /training first")

## 1. Carregar o manifest

**Objetivo:** ter um `DataFrame` com uma linha por amostra.

- Leia `MANIFEST_PATH` com pandas.
- Crie a coluna alvo `y` (1 se `label == POSITIVE_LABEL`, senão 0).
- Decida se vai usar o **frame inteiro** (`full_path`) ou o **recorte da mão** (`crop_path`). O recorte só existe quando `has_hand == 1`.

In [ ]:
# TODO

## 2. Explorar os dados

**Objetivo:** conhecer o dataset antes de modelar (cap. 2, *"Discover and visualize the data"*).

Perguntas para responder:
- As classes estão balanceadas? Quantas amostras há por **sessão**?
- Em quantos frames `has_hand` é 0? Isso muda muito entre as classes? (Se o MediaPipe perde a mão com mais frequência quando ela segura o handgrip, a própria coluna `has_hand` já é um sinal, e um vazamento em potencial.)
- Mostre algumas imagens de cada classe numa grade (`plt.subplots` + `Image.open`).

In [ ]:
# TODO

## 3. Separar o conjunto de teste (por sessão!)

**Objetivo:** reservar o teste **antes** de olhar mais a fundo (cap. 2, *"Create a test set"*).

⚠️ Frames da mesma sessão são quase idênticos. Um split aleatório por frame vaza informação e infla as métricas.
Separe **sessões inteiras**: veja `GroupShuffleSplit` em `sklearn.model_selection`, com `groups = df["session"]`.

Guarde algo como 20% das sessões para teste e **não mexa nelas** até a seção 11.

In [ ]:
# TODO

## 4. Features a partir dos pixels

**Objetivo:** transformar cada imagem num vetor, como no MNIST (cap. 3).

Sugestão: abrir → converter para tons de cinza (`.convert("L")`) → redimensionar para `IMAGE_SIZE` → `np.asarray(...)` → achatar → dividir por 255.
O resultado deve ser `X_train` com formato `(n_amostras, 1024)`.

Mostre uma imagem já processada com `plt.imshow(x.reshape(32, 32), cmap="gray")` para conferir.

In [ ]:
# TODO

## 5. Baseline

**Objetivo:** saber qual é o piso (cap. 3, o classificador *"never 5"*).

Um `DummyClassifier(strategy="most_frequent")` alcança qual acurácia? Todo modelo seguinte precisa superar esse número, e é por isso que acurácia sozinha engana.

In [ ]:
# TODO

## 6. `SGDClassifier` + validação cruzada por grupo

**Objetivo:** primeiro modelo de verdade (cap. 3).

- Monte um `Pipeline` com `StandardScaler` e `SGDClassifier`.
- Use `cross_val_predict` com `cv=GroupKFold(n_splits=...)` e `groups=` as sessões do treino.
- Pergunta: o resultado muda muito se você usar `cv=3` (split aleatório)? Por quê?

In [ ]:
# TODO

## 7. Métricas

**Objetivo:** ir além da acurácia (cap. 3, *"Performance measures"*).

- `confusion_matrix`, `precision_score`, `recall_score`, `f1_score`.
- Use `decision_function` (via `cross_val_predict(..., method="decision_function")`) para traçar a **curva precisão/revocação** e a **curva ROC** (`roc_auc_score`).
- Qual limiar faz sentido para o app? Errar para qual lado é pior: falso positivo ou falso negativo?

In [ ]:
# TODO

## 8. Comparar outros modelos

**Objetivo:** ver como algoritmos diferentes se saem nos mesmos dados e com os mesmos folds.

- `LogisticRegression` (cap. 4). Vai ser a mais fácil de exportar para o navegador.
- `SVC` com kernel RBF (cap. 5).
- `RandomForestClassifier` (cap. 7). Ele não precisa de escala; por quê?

Monte uma tabela com F1 e ROC AUC de cada um.

In [ ]:
# TODO

## 9. PCA

**Objetivo:** reduzir 1024 dimensões (cap. 8).

- Quantos componentes preservam 95% da variância (`PCA(n_components=0.95)`)?
- Visualize os primeiros componentes como imagens 32×32 (as "eigen-hands").
- Coloque o PCA no `Pipeline` e compare o desempenho e o tempo de treino.

In [ ]:
# TODO

## 10. Ajuste fino

**Objetivo:** buscar hiperparâmetros (cap. 2, *"Fine-tune your model"*).

`GridSearchCV` ou `RandomizedSearchCV` sobre o melhor pipeline, **com o mesmo `GroupKFold`**, passando `groups=` no `.fit`.

In [ ]:
# TODO

## 11. Avaliação final no teste

**Objetivo:** uma única medida honesta de generalização.

Treine o melhor pipeline com todo o treino e avalie **uma vez** nas sessões de teste. Se o resultado for bem pior que a validação cruzada, investigue o motivo antes de ajustar mais.

In [ ]:
# TODO

## 12. Exportar para o app

**Objetivo:** rodar o modelo no navegador, em TypeScript.

Para um pipeline `StandardScaler` → (`PCA`) → `LogisticRegression`, a predição é só álgebra linear:
salve `mean_`, `scale_`, `components_`, `coef_`, `intercept_` e o limiar escolhido em JSON dentro de `MODELS_DIR`.
No TS: `sigmoid(w · ((x - mean) / scale) + b) > limiar`.

Confira reimplementando a predição com NumPy e comparando com `predict_proba`.

In [ ]:
# TODO